#### This notebook is for logistic regression

In [2]:
library(rsample)     # data splitting 
library(dplyr)       # data wrangling
library(rpart)       # performing regression trees
library(rpart.plot)  # plotting regression trees
library(ipred)       # bagging
library(caret)       # bagging
library(smotefamily)
library(janitor) 
library(readr)
library(tidyverse)
library(caret)
library(xgboost)
library(glmnet)

Warning message:
"Paket 'rsample' wurde unter R Version 4.4.3 erstellt"

Attache Paket: 'dplyr'


Die folgenden Objekte sind maskiert von 'package:stats':

    filter, lag


Die folgenden Objekte sind maskiert von 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"Paket 'caret' wurde unter R Version 4.4.1 erstellt"
Lade n"otiges Paket: ggplot2

Warning message:
"Paket 'ggplot2' wurde unter R Version 4.4.3 erstellt"
Lade n"otiges Paket: lattice


Attache Paket: 'caret'


Das folgende Objekt ist maskiert 'package:rsample':

    calibration


Warning message:
"Paket 'janitor' wurde unter R Version 4.4.1 erstellt"

Attache Paket: 'janitor'


Die folgenden Objekte sind maskiert von 'package:stats':

    chisq.test, fisher.test


Warning message:
"Paket 'purrr' wurde unter R Version 4.4.3 erstellt"
Warning message:
"Paket 'forcats' wurde unter R Version 4.4.1 erstellt"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v forcats   1.0.1

In [16]:
set.seed(100)

df <- read_csv("data/merged_data.csv") |> clean_names() 

# outbreak >=2 is with outbreak, vice versa.
df$outbreak <- as.integer(df$outbreak >= 3)

# split the data with stratified sampling
strata <- ifelse(df$outbreak > 0, "nonzero", "zero")
index <- createDataPartition(strata, p = 0.75, list = FALSE)
train <- df[index,]
test <- df[-index,]
head(train)

train <- train[, colSums(is.na(train)) == 0]
train <- train[, !names(train) %in% c("county")]

# K for controlling the neighborhood, dup_size for ratio to achieve
smote_output <- SMOTE(
  X      = train[, names(train) != "outbreak"],
  target = train$outbreak,
  K      = 5, 
  dup_size = 0 
)

train_balanced <- smote_output$data
# SMOTE put target data in "class" col and rename it
names(train_balanced)[names(train_balanced) == "class"] <- "outbreak"
train_balanced$outbreak <- as.factor(train_balanced$outbreak)
cat("\nClass balance after SMOTE:\n")
print(table(train_balanced$outbreak))

# mirror on test set
test <- test[, colSums(is.na(test)) == 0]
test <- test[, !names(test) %in% c("county")]
test$outbreak <- as.factor(test$outbreak)

Rows: 254 Columns: 404
-- Column specification --------------------------------------------------------
Delimiter: ","
chr   (1): County
dbl (380): cve, outbreak, enrollment, PHR, pct_hispanic, pct_black, pct_whit...
lgl  (23): median_income, Advised to Cut Down Salt - Do not use salt, Diabet...

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


county,cve,outbreak,enrollment,phr,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,...,up_to_date_cervical_cancer_s_no,up_to_date_cervical_cancer_s_yes,up_to_date_cervical_cancer_s_2_no,up_to_date_cervical_cancer_s_2_yes,v_col_5_yrs_50_75_ia_no,v_col_5_yrs_50_75_ia_yes,v_col_5_yrs_age_50_75_no,v_col_5_yrs_age_50_75_yes,virtual_colonoscopy_no,virtual_colonoscopy_yes
<chr>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Anderson,2.54,0,7808,4,19.6,18.6,58.3,13.5,18.5,...,NA,NA,NA,NA,92.8,NA,92.8,NA,72.4,NA
Angelina,2.50,0,15649,5,23.5,12.1,63.4,11.5,17.7,...,36.9,63.1,36.8,63.2,92.2,NA,92.3,NA,65.5,34.5
Armstrong,5.24,0,297,1,12.0,0.6,89.9,6.0,4.2,...,39.9,60.1,40.0,60.0,96.9,NA,96.9,NA,76.0,NA
Austin,3.55,0,6290,6,27.8,7.9,66.6,10.6,16.7,...,39.2,60.8,39.1,60.9,97.0,NA,97.0,NA,74.3,25.7
Bailey,0.80,0,1330,1,66.0,0.9,71.3,7.4,28.6,...,39.9,60.1,40.0,60.0,96.9,NA,96.9,NA,76.0,NA
Bastrop,2.09,0,21555,7,45.0,5.8,54.2,7.0,21.8,...,29.7,70.3,30.1,69.9,97.6,NA,97.7,NA,79.9,20.1



Class balance after SMOTE:

  0   1 
177 168 


In [17]:
train_LR <- glm(outbreak ~ ., data = train_balanced, family = binomial)
train_LR_back <- step(train_LR,trace=0, direction = "backward")

Warning message:
"glm.fit: Angepasste Wahrscheinlichkeiten mit numerischem Wert 0 oder 1 aufgetreten"
Warning message:
"glm.fit: Angepasste Wahrscheinlichkeiten mit numerischem Wert 0 oder 1 aufgetreten"
Warning message:
"glm.fit: Algorithmus konvergierte nicht"
Warning message:
"glm.fit: Angepasste Wahrscheinlichkeiten mit numerischem Wert 0 oder 1 aufgetreten"
Warning message:
"glm.fit: Angepasste Wahrscheinlichkeiten mit numerischem Wert 0 oder 1 aufgetreten"
Warning message:
"glm.fit: Angepasste Wahrscheinlichkeiten mit numerischem Wert 0 oder 1 aufgetreten"
Warning message:
"glm.fit: Angepasste Wahrscheinlichkeiten mit numerischem Wert 0 oder 1 aufgetreten"
Warning message:
"glm.fit: Angepasste Wahrscheinlichkeiten mit numerischem Wert 0 oder 1 aufgetreten"
Warning message:
"glm.fit: Angepasste Wahrscheinlichkeiten mit numerischem Wert 0 oder 1 aufgetreten"
Warning message:
"glm.fit: Angepasste Wahrscheinlichkeiten mit numerischem Wert 0 oder 1 aufgetreten"
Warning message:
"glm.f

In [ ]:
lr_pred <- as.factor(predict(train_LR_back, newdata = test, type = "response"))
lr_pred <- as.numeric(as.character(lr_pred))
lr_pred <- ifelse(lr_pred > 0.5, 1, 0)
y_test  <- as.numeric(as.character(test$outbreak))
boost_table <- table(Actual = as.factor(y_test), Predicted = lr_pred)
boost_table

      Predicted
Actual  0  1
     0 36 21
     1  4  1